In [1]:
import numpy as np
import pandas as pd
from app.data.database import get_engine, canonical_rev_query
from app.core.config import settings


In [2]:
engine = get_engine()
# local_engine = get_engine(True)

selects = f"""
SELECT
    ams.invoice_date,
    amsl.quantity, amsl.price_subtotal, amsl.nepa_cogs_price, amsl.discount, amsl.standard_price, amsl.price_unit,
    CASE WHEN ams.journal_id in ({settings.DB_van_journals}) 
        THEN 'Van_:' || ams.journal_id::text
        ELSE 'Warehouse_:' || wss.warehouse_id::text
    END AS warehouse_label,
    wss.warehouse_id, wss.journal_id, wss.company_id, ams.partner_id, amsl.product_id, ams.move_type
"""
chunks = []
dates = ["2023-12-01", "2024-12-01", "2025-12-01", "2026-09-13"]
for i in range(len(dates)-1):
    query = canonical_rev_query(selects, start_date= dates[i], end_date=dates[i+1])

    print("connecting to nepa db")
    for chunk in pd.read_sql(query, engine, chunksize=500000):
        chunks.append(chunk)
        print(f"Loaded chunk with {len(chunk)} rows")
    
engine.dispose()

df_kpi_forecasting: pd.DataFrame = pd.concat(chunks, ignore_index=True)

connected successfully; version PostgreSQL 17.11 (Ubuntu 17.11-1.pgdg22.04+2) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0, 64-bit
connecting to nepa db
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 38902 rows
connecting to nepa db
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 373589 rows
connecting to nepa db
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 500000 rows
Loaded chunk with 437151 rows


In [3]:
df_kpi_forecasting["invoice_date"] = pd.to_datetime(
    df_kpi_forecasting["invoice_date"]
)

In [4]:
df_features = (
    df_kpi_forecasting.groupby("invoice_date", as_index=False)
    .agg(
        nepa_cogs_price_sum=("nepa_cogs_price", "sum"),
        price_unit_sum=("price_unit", "sum"),
        price_unit_mean=("price_unit", "mean"),
        revenue_signed_sum=("revenue_signed", "sum"),
        qty_signed_sum=("qty_signed", "sum"),
        discount_sum=("discount", "sum"),
        product_id_unique=("product_id", "nunique"),
        active_customer=("partner_id", "nunique"),
        order_count=("move_type", lambda x: (x == "out_invoice").sum()),
        refund_count=("move_type", lambda x: (x == "out_refund").sum()),
    )
)

In [5]:
df_features.columns

Index(['invoice_date', 'nepa_cogs_price_sum', 'price_unit_sum',
       'price_unit_mean', 'revenue_signed_sum', 'qty_signed_sum',
       'discount_sum', 'product_id_unique', 'active_customer', 'order_count',
       'refund_count'],
      dtype='str')

In [6]:
df_features = df_features.sort_values(by="invoice_date").reset_index(drop=True)

full_date_range = pd.date_range(
    start=df_features["invoice_date"].min(),
    end=df_features["invoice_date"].max(),
    freq="D"
)

df_features = (
    df_features.set_index('invoice_date')
    .reindex(full_date_range)
    .ffill()
    .rename_axis("invoice_date")
    # .reset_index()
)

In [8]:
# date features
df_features["invoice_date"] = df_features.index

df_features["day_of_week"] = df_features["invoice_date"].dt.day_name()

 # revenue target
df_features["target_rev"] = df_features["revenue_signed_sum"].shift(-1)

# qty target
df_features["target_qty"] = df_features["qty_signed_sum"].shift(-1)

# active cust tgt
df_features["target_cust"] = df_features["active_customer"].shift(-1)

# order target
df_features["target_order"] = df_features["order_count"].shift(-1)

# refund target
df_features["target_refund"] = df_features["refund_count"].shift(-1)




# week of day features
dow_sma = [5,6,7,8]
for n_week in dow_sma:
    df_features[f"day_of_week_sma{n_week}W_rev"] = (df_features.groupby("day_of_week"))["target_rev"].transform(lambda x: x.shift(1).rolling(window=n_week, min_periods=1).mean())

dow_sma = [5,6,7,8]
for n_week in dow_sma:
    df_features[f"day_of_week_sma{n_week}W_qty"] = (df_features.groupby("day_of_week"))["target_qty"].transform(lambda x: x.shift(1).rolling(window=n_week, min_periods=1).mean())

dow_sma = [5,6,7,8]
for n_week in dow_sma:
    df_features[f"day_of_week_sma{n_week}W_cust"] = (df_features.groupby("day_of_week"))["target_cust"].transform(lambda x: x.shift(1).rolling(window=n_week, min_periods=1).mean())

dow_sma = [5,6,7,8]
for n_week in dow_sma:
    df_features[f"day_of_week_sma{n_week}W_order"] = (df_features.groupby("day_of_week"))["target_order"].transform(lambda x: x.shift(1).rolling(window=n_week, min_periods=1).mean())

dow_sma = [5,6,7,8]
for n_week in dow_sma:
    df_features[f"day_of_week_sma{n_week}W_refund"] = (df_features.groupby("day_of_week"))["target_refund"].transform(lambda x: x.shift(1).rolling(window=n_week, min_periods=1).mean())



# SMA features and lag
cols = ["qty_signed_sum", "revenue_signed_sum", "active_customer", "order_count", "refund_count"]
prefix_map = {
    "revenue_signed_sum" : "rev",
    "qty_signed_sum" : "qty", 
    "active_customer": "cust",
    "order_count": "order",
    "refund_count": "refund"
}
sma_duration = [7,14,28]
for col in cols:
    prefix = prefix_map.get(col,col)
    df_features[F"{prefix}_lag_7"] = df_features[col].shift(6)
    for i in sma_duration:
        df_features[f"{prefix}_{i}d_ma"] = (df_features[col].shift(1).rolling(window=i, min_periods=1).mean())



In [18]:
import plotly.express as px
import plotly.subplots as sp
import plotly.graph_objects as go

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 100)

cates = {
    'rev': 'target_rev',
    'qty': 'target_qty',
    'cust': 'target_cust',
    'order': 'target_order',
    'refund': 'target_refund'
}


for col in cols:
    prefix = prefix_map.get(col, col)

    ind_cols = [col for col in df_features.columns  if prefix in col and 'sma' in col and 'target' not in col]

    fig = sp.make_subplots(rows=4, cols=1, subplot_titles=[f"{title} vs target" for title in ind_cols])

    fig.update_layout(
        colorway=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#970505"]
)
    
    for idx, column in enumerate(ind_cols, start=1):
        fig.add_trace(
            go.Scatter(x=df_features["invoice_date"], y=df_features[cates[prefix]], name= f"{i}d MA"),
            col=1, row=idx
        )

        fig.add_trace(
            go.Scatter(x=df_features["invoice_date"], y=df_features[column], name= f"dow sma MA"),
            col=1, row=idx
        )
    
    fig.update_layout(height=1300, title_text=f"MA of {col}")
    fig.show()

In [14]:
df_features = df_features.iloc[6:]

In [15]:
feature_cols = ["revenue_signed_sum", "qty_signed_sum", "active_customer", "order_count", "refund_count"]

In [1]:
# Rolling backtest harness

# 1. Ensure datetime type and sort by date
df_features['invoice_date'] = pd.to_datetime(df_features['invoice_date'])
df_features = df_features.sort_index()

min_date = df_features['invoice_date'].min()
max_date = df_features['invoice_date'].max()


if min_date.weekday() == 6:  #sunday 
    first_full_week_start = min_date.normalize()
else:
    first_full_week_start = (min_date + pd.offsets.Week(weekday=6)).normalize()


if max_date.weekday() == 5:  # saturday
    last_full_week_end = max_date
else:
    last_full_week_end = (max_date - pd.offsets.Week(weekday=5)).replace(
        hour=23, minute=59, second=59
    )

df_backtest = df_features[
    (df_features['invoice_date'] >= first_full_week_start)
    & (df_features['invoice_date'] <= last_full_week_end)
].copy()

# X_btest = df_backtest[feature_cols].drop(["target", "target_qty"], axis=1)
# y_btest = df_backtest[target_col],

def wape(tgt, pred):
    sum_actuals = np.sum(np.abs(tgt))
    wape_val = np.sum(np.abs(tgt - pred)) / sum_actuals if sum_actuals != 0 else np.nan
    return wape_val
    


NameError: name 'pd' is not defined

In [ ]:
test_week_starts = pd.date_range(
    start = first_full_week_start,
    end=last_full_week_end-pd.Timedelta(days=6),
    freq="7D"
)
rev_score = 0 # to calc how many slices the model was better than naive
qty_score = 0
week_num = 0



for week_start in test_week_starts[:-2]:
    week_end = week_start +pd.Timedelta(days=27) # set slice window
    if week_start == pd.Timestamp('2026-04-12'):
        week_end = week_start +pd.Timedelta(days=6) # set slice window
    week_num+= 1
    pred_hist = pd.DataFrame()
    for date in pd.date_range(week_start, week_end, freq="D"):
        pred = pd.DataFrame(
            [{
                'invoice_date': date,

                'day_of_week_sma5W_rev': df_features.loc[date, "day_of_week_sma5W_rev"],
                'day_of_week_sma6W_rev': df_features.loc[date, "day_of_week_sma6W_rev"],
                'day_of_week_sma7W_rev': df_features.loc[date, "day_of_week_sma7W_rev"],
                'day_of_week_sma8W_rev': df_features.loc[date, "day_of_week_sma8W_rev"],
                'rev_lag_7': df_features.loc[date, "rev_lag_7"],
                'rev_7d_ma': df_features.loc[date, "rev_7d_ma"],
                'rev_14d_ma': df_features.loc[date, "rev_14d_ma"],
                'rev_28d_ma': df_features.loc[date, "rev_28d_ma"],
                'target': df_features.loc[date, "target"],

                'day_of_week_sma5W_qty': df_features.loc[date, "day_of_week_sma5W_qty"],
                'day_of_week_sma6W_qty': df_features.loc[date, "day_of_week_sma6W_qty"],
                'day_of_week_sma7W_qty': df_features.loc[date, "day_of_week_sma7W_qty"],
                'day_of_week_sma8W_qty': df_features.loc[date, "day_of_week_sma8W_qty"],
                'qty_lag_7': df_features.loc[date, "qty_lag_7"],
                'qty_7d_ma': df_features.loc[date, "qty_7d_ma"],
                'qty_14d_ma': df_features.loc[date, "qty_14d_ma"],
                'qty_28d_ma': df_features.loc[date, "qty_28d_ma"],
                'target_qty': df_features.loc[date, "target_qty"],

                'day_of_week_sma5W_cust': df_features.loc[date, "day_of_week_sma5W_cust"],
                'day_of_week_sma6W_cust': df_features.loc[date, "day_of_week_sma6W_cust"],
                'day_of_week_sma7W_cust': df_features.loc[date, "day_of_week_sma7W_cust"],
                'day_of_week_sma8W_cust': df_features.loc[date, "day_of_week_sma8W_cust"],
                'cust_lag_7': df_features.loc[date, "cust_lag_7"],
                'cust_7d_ma': df_features.loc[date, "cust_7d_ma"],
                'cust_14d_ma': df_features.loc[date, "cust_14d_ma"],
                'cust_28d_ma': df_features.loc[date, "cust_28d_ma"],
                'target_cust': df_features.loc[date, "target_cust"],

                'day_of_week_sma5W_order': df_features.loc[date, "day_of_week_sma5W_order"],
                'day_of_week_sma6W_order': df_features.loc[date, "day_of_week_sma6W_order"],
                'day_of_week_sma7W_order': df_features.loc[date, "day_of_week_sma7W_order"],
                'day_of_week_sma8W_order': df_features.loc[date, "day_of_week_sma8W_order"],
                'order_lag_7': df_features.loc[date, "order_lag_7"],
                'order_7d_ma': df_features.loc[date, "order_7d_ma"],
                'order_14d_ma': df_features.loc[date, "order_14d_ma"],
                'order_28d_ma': df_features.loc[date, "order_28d_ma"],
                'target_order': df_features.loc[date, "target_order"],

                'day_of_week_sma5W_refund': df_features.loc[date, "day_of_week_sma5W_refund"],
                'day_of_week_sma6W_refund': df_features.loc[date, "day_of_week_sma6W_refund"],
                'day_of_week_sma7W_refund': df_features.loc[date, "day_of_week_sma7W_refund"],
                'day_of_week_sma8W_refund': df_features.loc[date, "day_of_week_sma8W_refund"],
                'refund_lag_7': df_features.loc[date, "refund_lag_7"],
                'refund_7d_ma': df_features.loc[date, "refund_7d_ma"],
                'refund_14d_ma': df_features.loc[date, "refund_14d_ma"],
                'refund_28d_ma': df_features.loc[date, "refund_28d_ma"],
                'target_refund': df_features.loc[date, "target_refund"],

            }],
            index = [date]
        )
        pred_hist = pd.concat([pred_hist,pred])
    # retrain model on new data

    rev_cols = [col for col in pred_hist.columns if 'rev' in col and 'target' not in col]
    qty_cols = [col for col in pred_hist.columns if 'qty' in col and 'target' not in col]
    cust_cols = [col for col in pred_hist.columns if 'cust' in col and 'target' not in col]
    order_cols = [col for col in pred_hist.columns if 'order' in col and 'target' not in col]
    refund_cols = [col for col in pred_hist.columns if 'refund' in col and 'target' not in col]

    cates = {
        'target': rev_cols,
        'target_qty': qty_cols,
        'target_cust': cust_cols,
        'target_order': order_cols,
        'target_refund': refund_cols
    }



    for target, cate in cates.items():
        for col in cate:
           result =  wape(pred_hist[target],pred_hist[col])
           print(f"{col} wape for window {week_start}: {result}")


            

        

    # print(f'rev wape: {wape_btest_rev}')
    # print(f'qty wape: {wape_btest_qty}')

print(week_num)
print((f"for week: {week_start} to {week_end}"))
print((f"rev pred by model is better than naive t-7 in {rev_score/week_num} of slices"))
print((f"qty pred by model is better than naive t-7 in {qty_score/week_num} of slices"))

day_of_week_sma5W_rev wape for window 2023-12-10 00:00:00: 0.47967912984711775
day_of_week_sma6W_rev wape for window 2023-12-10 00:00:00: 0.47967912984711775
day_of_week_sma7W_rev wape for window 2023-12-10 00:00:00: 0.47967912984711775
day_of_week_sma8W_rev wape for window 2023-12-10 00:00:00: 0.47967912984711775
rev_lag_7 wape for window 2023-12-10 00:00:00: 0.503169702915297
rev_7d_ma wape for window 2023-12-10 00:00:00: 0.6153363153900366
rev_14d_ma wape for window 2023-12-10 00:00:00: 0.6195873112684006
rev_28d_ma wape for window 2023-12-10 00:00:00: 0.6012473767721096
day_of_week_sma5W_qty wape for window 2023-12-10 00:00:00: 0.4823326957076196
day_of_week_sma6W_qty wape for window 2023-12-10 00:00:00: 0.4823326957076196
day_of_week_sma7W_qty wape for window 2023-12-10 00:00:00: 0.4823326957076196
day_of_week_sma8W_qty wape for window 2023-12-10 00:00:00: 0.4823326957076196
qty_lag_7 wape for window 2023-12-10 00:00:00: 0.4743500565168246
qty_7d_ma wape for window 2023-12-10 00:0

KeyError: Timestamp('2026-09-04 00:00:00')